In [1]:
import pandas as pd
import sys

sys.path.insert(0, '..') # Config-driven
from config import settings

report = pd.read_csv(settings.OUTPUT_DIR / 'validation_report.csv')
clean = pd.read_csv(settings.OUTPUT_DIR / 'valid_transactions.csv')
flagged = pd.read_csv(settings.OUTPUT_DIR / 'flagged_issues.csv')

Part 1: Pattern Matching — .str.contains()

In [2]:
# Which transactions have LEI004 in their error codes?
mask = flagged['error_codes'].str.contains('LEI004')
mask.head(10)

0     True
1     True
2     True
3    False
4     True
5     True
6     True
7    False
8     True
9     True
Name: error_codes, dtype: bool

In [3]:
# Filter: only transactions with LEI004
lei004_txns = flagged[flagged['error_codes'].str.contains('LEI004')]
print(f"Transactions with LEI004: {len(lei004_txns)} out of {len(flagged)}")

# Filter: flagged transactions WITHOUT LEI004
no_lei004 = flagged[~flagged['error_codes'].str.contains('LEI004')]
print(f"Transactions WITHOUT LEI004: {len(no_lei004)} out of {len(flagged)}")

Transactions with LEI004: 78 out of 128
Transactions WITHOUT LEI004: 50 out of 128


In [4]:
# What if error_codes has NaN? (clean transactions have no errors)
report['error_codes'].str.contains('LEI004').tail(10)

# Defensive version — na=False means "if the value is NaN, return False"
report['error_codes'].str.contains('LEI004', na=False)

0      False
1      False
2      False
3       True
4       True
       ...  
195    False
196    False
197    False
198    False
199    False
Name: error_codes, Length: 200, dtype: bool

In [5]:
report['error_codes'].str.contains('LEI004').value_counts(dropna=False)

error_codes
False    122
True      78
Name: count, dtype: int64

Pattern Matching — str.startswith()

In [6]:
# Which issuing banks have Swift codes starting with 'DE' (German banks)?
german_banks = report.query("issuing_bank_swift.str.startswith('DE')")
print(f"German issuing banks: {len(german_banks)}")
german_banks[['transaction_id', 'issuing_bank_name', 'issuing_bank_swift']].head()

German issuing banks: 27


,transaction_id,issuing_bank_name,issuing_bank_swift
8,LC-3J2D54,Deutsche Bank,DEUTDEFF
17,LC-MLY4LY,Deutsche Bank,DEUTDEFF
29,LC-HAIR4C,Deutsche Bank,DEUTDEFF
30,LC-YJXRPJ,Deutsche Bank,DEUTDEFF
36,LC-A5U6HC,Deutsche Bank,DEUTDEFF


Pattern Matching —  .str.endswith():

In [7]:
# Which LC numbers end with a letter (not a digit)?
# [A-Za-z] would be regex, but endswith is literal — so let's use contains with regex
letter_ending = report[report['lc_number'].str.contains(r'[A-Za-z]$')]
print(f"LC numbers ending with a letter: {len(letter_ending)}")
letter_ending['lc_number'].head(10)


LC numbers ending with a letter: 5


31                            LC-XXXXXXXXXXXXXXXXX
48                                              LC
90                                              LC
112    LC-XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
138                                             LC
Name: lc_number, dtype: str

Part 2: Extracting Data — .str.extract()

In [8]:
# Extract country code from SWIFT (positions 4-5, which is characters 5-6)
report['issuing_bank_swift_country'] = (
    report['issuing_bank_swift'].str.extract(r'^.{4}([A-Z]{2})')
)
report[['issuing_bank_swift', 'issuing_bank_swift_country']].drop_duplicates().head(10)

,issuing_bank_swift,issuing_bank_swift_country
0,SMBCJPJT,JP
1,WFBIUS6S,US
2,CHASUS33,US
3,CZNBKRSE,KR
4,CTCBTWTP,TW
7,BOFAUS3N,US
8,DEUTDEFF,DE
9,BOTKJPJT,JP
11,BARCGB22,GB
12,SHBKKRSE,KR


In [9]:
# Which countries issue the most LCs?
report['issuing_bank_swift_country'].value_counts()

issuing_bank_swift_country
US    76
KR    29
JP    27
DE    27
GB    19
TW    13
Name: count, dtype: int64

Notice: 76 + 29 + 27 + 27 + 19 + 13 = 191, not 200. Those missing 9 are probably transactions with bad/missing SWIFT codes (remember you have 6 BIC001 errors + 3 BIC002). The extract returned NaN for those and value_counts() skips NaN by default.

Part 3: Splitting — .str.split() and .str.get()

In [10]:
# Split error_codes into a list
flagged['error_codes'].str.split(', ').head(5)

# each cell is now a Python list instead of a string

0    [LEI004, DATE007]
1    [LEI004, SHIP002]
2             [LEI004]
3            [DATE002]
4    [LEI004, XVAL002]
Name: error_codes, dtype: object

 .str.get()

In [11]:
# Get the FIRST error code for each flagged transaction
flagged['error_codes'].str.split(', ').str.get(0).head(10)

# That's chaining: .str.split() makes lists → .str.get(0) grabs the first item from each list

0     LEI004
1     LEI004
2     LEI004
3    DATE002
4     LEI004
5     LEI004
6     LEI004
7    DATE007
8     LEI004
9     LEI004
Name: error_codes, dtype: object

split into separate columns

In [12]:
# expand=True → each piece gets its own column
flagged['error_codes'].str.split(', ', expand=True).head(10)

,0,1,2,3,4,5
0,LEI004,DATE007,NaN,NaN,NaN,NaN
1,LEI004,SHIP002,NaN,NaN,NaN,NaN
2,LEI004,NaN,NaN,NaN,NaN,NaN
3,DATE002,NaN,NaN,NaN,NaN,NaN
4,LEI004,XVAL002,NaN,NaN,NaN,NaN
5,LEI004,NaN,NaN,NaN,NaN,NaN
6,LEI004,LC005,NaN,NaN,NaN,NaN
7,DATE007,NaN,NaN,NaN,NaN,NaN
8,LEI004,NaN,NaN,NaN,NaN,NaN
9,LEI004,NaN,NaN,NaN,NaN,NaN


Part 4: Replacing — .str.replace()

In [13]:
# Remove the severity prefix from error_messages
# Before: "[CRITICAL] AMT005: amount - Invalid amount format (got: abc)"
# After:  "AMT005: amount - Invalid amount format (got: abc)"
sample = flagged['error_messages'].str.replace(r'\[(CRITICAL|HIGH|MEDIUM|LOW)\]\s*', '', regex=True)
sample.head(3)

0    LEI004: applicant_lei - LEI not found in GLEIF...
1    LEI004: applicant_lei - LEI not found in GLEIF...
2    LEI004: applicant_lei - LEI not found in GLEIF...
Name: error_messages, dtype: str

Part 5: Measuring — .str.len()

In [14]:
# How long are the LC numbers?
report['lc_number_length'] = report['lc_number'].str.len()
report['lc_number_length'].describe()

count    198.000000
mean      11.070707
std        2.614883
min        2.000000
25%       11.000000
50%       11.000000
75%       11.000000
max       43.000000
Name: lc_number_length, dtype: float64

In [15]:
## Exercise

In [16]:
# Step 1: Explode error_codes into one row per error
errors_exploded = (
    flagged['error_codes']
    .str.split(', ') #  — splits each string into a list (no expand=True this time): 'LEI001, LEI004'   →  ['LEI001', 'LEI004']
    .explode() #  .explode() — unpacks each list into separate rows:
)
errors_exploded.head(10)

0     LEI004
0    DATE007
1     LEI004
1    SHIP002
2     LEI004
3    DATE002
4     LEI004
4    XVAL002
5     LEI004
6     LEI004
Name: error_codes, dtype: str

In [17]:
 errors_exploded.value_counts()  # how many times did each error code appear?

error_codes
LEI004     86
SHIP002    23
XVAL002    20
SHIP005    12
AMT002     11
DATE005     7
AMT005      7
BIC001      6
DATE007     4
LC005       4
AMT006      4
XVAL001     4
SHIP006     3
SHIP004     3
LC002       3
AMT001      3
BIC002      3
LEI001      3
PTY002      3
SHIP001     3
DATE002     2
LEI002      2
LC001       2
AMT003      2
LC007       2
PTY004      2
DATE004     2
BIC003      2
DATE001     2
PTY005      1
LC008       1
LC004       1
LC003       1
PTY001      1
LEI003      1
SHIP003     1
Name: count, dtype: int64

In [18]:
# Step 2: Extract the letter prefix (AMT, LEI, BIC, DATE, etc.)
error_categories = errors_exploded.str.extract(r'^([A-Z]+)')[0]
error_categories.value_counts()

#  If you had two groups like r'^([A-Z]+)_(\d+)', the result would have columns 0 and 1.

0
LEI     92
SHIP    45
AMT     27
XVAL    24
DATE    17
LC      14
BIC     11
PTY      7
Name: count, dtype: int64

1-- "Which validator causes the most CRITICAL errors?"

In [19]:
## INCORRECT
# critical = flagged[flagged['severity'] == 'CRITICAL']
critical = flagged.query("severity == 'CRITICAL'")
critical_cat = (

    critical['error_codes']
    .str.split(', ')
    .explode()
    .str.extract(r'^([A-Z]+)')[0]

)

critical_cat.value_counts()

0
LEI     28
SHIP    25
AMT     21
LC      14
DATE    10
XVAL     9
BIC      2
PTY      2
Name: count, dtype: int64

In [20]:
from config.validation_rules import ALL_ERROR_CODES
## WAy 1
# Build a mapping: error_code → severity
severity_map = {code: info['severity'] for code, info in ALL_ERROR_CODES.items()}

# Now: explode ALL errors, map their TRUE severity, then filter
all_errors = flagged['error_codes'].str.split(', ').explode()
all_errors_severity = all_errors.map(severity_map)


# Combine into a DataFrame
error_detail = pd.DataFrame({
    'error_code': all_errors,
    'severity': all_errors_severity,
    'category': all_errors.str.extract(r'^([A-Z]+)')[0]
})

# NOW filter for truly CRITICAL errors
error_detail.query("severity == 'CRITICAL'")['category'].value_counts()

category
AMT     21
LC      14
DATE     9
PTY      1
Name: count, dtype: int64

In [21]:
## WAY 2
# Explode error_messages, then extract the severity from each one
msg_exploded = flagged['error_messages'].str.split(', ').explode()
error_analysis = pd.DataFrame({
    'severity': msg_exploded.str.extract(r'\[(CRITICAL|HIGH|MEDIUM|LOW)\]')[0],
    'category': msg_exploded.str.extract(r'\]\s*([A-Z]+)\d')[0]
})

error_analysis.query("severity == 'CRITICAL'")['category'].value_counts()

category
AMT     21
LC      14
DATE     9
PTY      1
Name: count, dtype: int64

In [22]:
all_err = flagged['error_codes'].str.split(', ').explode()
all_err.value_counts()
all_err.head(10)

0     LEI004
0    DATE007
1     LEI004
1    SHIP002
2     LEI004
3    DATE002
4     LEI004
4    XVAL002
5     LEI004
6     LEI004
Name: error_codes, dtype: str

2 _give me all the transactions which have 'SHIP006' or 'AMT005' but onlt show me issue dates

In [23]:
 flagged[flagged['error_codes'].str.contains('SHIP006|AMT005')]['issue_date']

10     2026-01-12
23     2026-01-15
26     2026-01-07
49     2026-01-20
68     2026-01-20
69     2026-01-22
80     2026-01-14
81     2026-01-11
82     2026-01-09
110    2026-01-25
Name: issue_date, dtype: str

In [24]:
 # Extract country code from SWIFT (positions 4-5, which is characters 5-6)
 # clean = report.query("validation_status == 'CLEAN'")
report['issuing_bank_swift_country'] = (
    report['issuing_bank_swift'].str.extract(r'^.{4}([A-Z]{2})')
)
total_per_country = report.groupby('issuing_bank_swift_country')['transaction_id'].count()

clean_per_country = report.query("validation_status == 'CLEAN'").groupby('issuing_bank_swift_country')['transaction_id'].count()

clean_rate = (clean_per_country / total_per_country * 100).sort_values(ascending=False)
clean_rate.round(1)



issuing_bank_swift_country
US    53.9
GB    52.6
JP    29.6
DE    25.9
TW    15.4
KR    13.8
Name: transaction_id, dtype: float64

In [25]:
pd.crosstab(report['issuing_bank_swift_country'], report['validation_status'], normalize='index').round(3) * 100

validation_status,CLEAN,FLAGGED
issuing_bank_swift_country,,
DE,25.9,74.1
GB,52.6,47.4
JP,29.6,70.4
KR,13.8,86.2
TW,15.4,84.6
US,53.9,46.1


In [26]:


flagged['err_AMT'] =( flagged['error_codes'].str.contains('AMT'))
flagged.groupby('currency')['err_AMT'].mean().round(2)
# Total errors per currency
# report.groupby('currency')['error_count'].sum()
#flagged['error_count_AMT'] = err_AMT.value_counts()
# Average errors per currency


# report.groupby('currency')['transaction_id'].count()

amt_txns = flagged[flagged['error_codes'].str.contains('AMT')]
amt_txns.groupby('currency')['error_count'].mean().round(2)


currency
EUR    1.00
GBP    3.89
JPY    2.33
USD    3.00
Name: error_count, dtype: float64

4__Find the top 5 transactions with the most diverse error types

In [27]:
# Not the most errors — the most different validator categories. A transaction with LEI004, LEI004, LEI001 has 3 errors but only 1 category (LEI). A transaction with LEI004, AMT005, DATE002 has 3 errors and 3 categories.
# Split → explode → extract category
# nunique() counts distinct values
# groupby
errors_exploded = (
    flagged['error_codes']
    .str.split(', ') #  — splits each string into a list (no expand=True this time): 'LEI001, LEI004'   →  ['LEI001', 'LEI004']
    .explode() #  .explode() — unpacks each list into separate rows:
)
error_cat = errors_exploded.str.extract(r'^([A-Z]+)')
diversity = (
  error_cat[0]
  .groupby(flagged['transaction_id'])   # pass the column directly
  .nunique()
  .sort_values(ascending=False)
  .rename("unique_error_types")
)

diversity.head()

error_cat = errors_exploded.str.extract(r'^([A-Z]+)')[0].groupby(flagged['transaction_id']).nunique().sort_values(ascending=False).rename("unique_error_types")
error_cat.head(5)

transaction_id
LC-D84U89    5
LC-0TPAC5    4
LC-CW790P    4
LC-3WXA73    4
LC-HE7UR2    4
Name: unique_error_types, dtype: int64

Challenge 5: "Build a risk summary — one row per currency with multiple stats"
Build a single DataFrame showing, per currency:

Total transactions
Number flagged
Flagged percentage
Average error count
Most common error code

In [28]:
# helper column first  then aggregate it. Boolean sum counts the True values:
report['is_flagged'] = report['validation_status'] == 'FLAGGED'

report.groupby('currency').agg(
    total_txn=('transaction_id', 'count'),
    flagged_number=('is_flagged', 'sum'),
    flagged_pct=('is_flagged', lambda x :(x.mean() *100)),
    avg_error_count=('error_count', 'mean'),
    # most_common_error = ('error_codes', lambda  x : x.dropna().str.split(', ').explode().value_counts().idxmax())
    most_common_error=('error_codes', lambda x:
    x[x != ''].str.split(', ').explode().value_counts().index[0]
    if x[x != ''].any() else 'NONE'
    )
).round(2)

# lambda lets u write custom logic inside
#  x.mean() on a boolean column = sum of Trues / count of all
# .mean() does exactly what it always does — sum divided by count:

# .agg() only accepts (column_name, aggregation_function) tuples — you can't pass a filtered DataFrame inside it.
#  The workaround is to encode what you want to count as a column (True/False), then sum it per group. True counts as 1, False as 0.

,total_txn,flagged_number,flagged_pct,avg_error_count,most_common_error
currency,,,,,
EUR,44,27,61.36,0.91,LEI004
GBP,47,28,59.57,1.43,LEI004
JPY,58,37,63.79,1.00,LEI004
USD,51,36,70.59,1.41,LEI004


In [29]:
# 1- Filter to transactions that contain at least one AMT error
# 2- Explode and extract ALL their error categories
# 3- Remove AMT from the results (we already know AMT is there)
# 4- Count what's left

# 1
filtered_txn =  flagged.loc[flagged["error_codes"].str.contains("AMT", na=False),( "transaction_id", "error_codes")]
error = filtered_txn['error_codes'].str.split(', ').explode().str.extract(r'^(?P<error_category>[A-Z]+)')
non_amt = error[error['error_category'] != 'AMT']
non_amt['error_category'].value_counts()
# flagged[flagged["error_codes"].str.contains("AMT", na=False)]["transaction_id"]

# 2
# error_codes ONCE  ( explode and extract ) YAP SONRA  error categorilerine ayir ( GROUPBY)
error = filtered_txn['error_codes'].str.split(', ').explode().str.extract(r'^(?P<error_category>[A-Z]+)')
# error = filtered_txn['error_codes'].str.split(', ').explode().str.extract(r'^([A-Z]+)')[0].rename('error_category')
# error.str.extract(r'^([A-Z]+)')
#error = filtered_txn['error_codes'].str.split(', ').explode().str.extract(r'^(?P<error_category>[A-Z]+)')
#err = error.groupby('error_category')['error_category'].count()

 # 3 - Remove AMT rows
non_amt = error[error['error_category'] != 'AMT']
 # 4 - Count what's left
non_amt['error_category'].value_counts()


"""
  1. Filter — keep only transactions that have at least one AMT error
  2. Explode — break the comma-separated error codes into individual rows (one error per row)
  3. Extract — pull out just the category prefix (e.g. AMT, DAT, PTY)
  4. Remove AMT — we already know AMT is there, so drop those rows
  5. Count — see what other error categories co-occur with AMT transactions
"""

'\n  1. Filter — keep only transactions that have at least one AMT error\n  2. Explode — break the comma-separated error codes into individual rows (one error per row)\n  3. Extract — pull out just the category prefix (e.g. AMT, DAT, PTY)\n  4. Remove AMT — we already know AMT is there, so drop those rows\n  5. Count — see what other error categories co-occur with AMT transactions\n'